[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/lin-elastic_strain.ipynb)

# Linear-Elastic Strain Solve

A minimal walkthrough of FFTjax's strain-based Newton-CG elastic solver (`solvers.mechanical.strain_nw_cg.solve_elastic`) on a **homogeneous, isotropic** cube under a prescribed macroscopic strain.

Because the material is homogeneous and the reference medium is chosen to match it exactly, the correction field is exactly zero — the local strain equals the prescribed macroscopic strain everywhere, and the local stress follows directly from Hooke's law. This makes the result easy to check by hand, so the notebook doubles as a sanity check of the install.

For a case where the solver actually iterates (a two-phase composite), see the [Benchmark](../benchmark.md#linear-elastic-strain-solve) page.

## Setup

In [1]:
import os
os.environ["JAX_ENABLE_X64"] = "1"
import sys
sys.path.insert(0, "../src")

import jax
import jax.numpy as jnp
import numpy as np

from operators.green import build_freq_grid, build_green_operator
from mat_models.elastic import LinearElasticIsotropic, assemble_C_field
from solvers.mechanical.strain_nw_cg import solve_elastic

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())

JAX backend: cpu
Devices: [CpuDevice(id=0)]

## Grid and material

A 32³ voxel unit cube, single-phase isotropic steel.

In [1]:
n = (32, 32, 32)
L = (1.0, 1.0, 1.0)
Nv = int(np.prod(n))

material = LinearElasticIsotropic(E=210e3, nu=0.3, name="steel")
phase = jnp.zeros(Nv, dtype=int)          # homogeneous: single phase everywhere
C_field = assemble_C_field([material], phase)

print(material)

LinearElasticIsotropic (steel): E=2.1e+05, nu=0.3, lam=1.21e+05, mu=8.08e+04

## Frequency grid and Green's operator

The reference medium (`lam0`, `mu0`) is set to the material's own Lamé parameters — exact for a homogeneous material.

In [1]:
xi_flat = build_freq_grid(n, L)
G_glob = build_green_operator(xi_flat, material.lam, material.mu)

print("xi_flat shape:", xi_flat.shape)
print("G_glob shape :", G_glob.shape)

xi_flat shape: (3, 32768)
G_glob shape : (3, 3, 3, 3, 32768)

## Prescribe a macroscopic strain and solve

A uniaxial strain of 10⁻³ along x.

In [1]:
eps_bar = jnp.array([
    [1.0e-3, 0.0, 0.0],
    [0.0,    0.0, 0.0],
    [0.0,    0.0, 0.0],
])

eps, sigma, delta, it, converged = solve_elastic(n, C_field, G_glob, eps_bar)

print("CG iterations:", int(it))
print("converged     :", bool(converged))

CG iterations: 0
converged     : True

## Check against Hooke's law

For a homogeneous material the local strain must equal `eps_bar` everywhere, and the local stress must equal the direct Hooke's-law prediction `C : eps_bar`.

In [1]:
sigma_analytic = material.stress_field(jnp.broadcast_to(eps_bar[:, :, None], (3, 3, Nv)))

max_err_eps   = float(jnp.max(jnp.abs(eps - eps_bar[:, :, None])))
max_err_sigma = float(jnp.max(jnp.abs(sigma - sigma_analytic)))

print(f"max |eps  - eps_bar|        = {max_err_eps:.3e}")
print(f"max |sigma - C:eps_bar|     = {max_err_sigma:.3e}")

assert max_err_eps   < 1e-10
assert max_err_sigma < 1e-8
print("\nPASSED — local fields match the analytic homogeneous solution.")

max |eps  - eps_bar|        = 0.000e+00
max |sigma - C:eps_bar|     = 0.000e+00

PASSED — local fields match the analytic homogeneous solution.

## Next steps

- Swap in a two-phase `C_field` (e.g. an inclusion) to see the Newton-CG solve actually   iterate — the [Benchmark](../benchmark.md#linear-elastic-strain-solve) page does exactly   this and times it across grid sizes.
- See `solvers.mechanical.strain_nw_cg.dstrain_nw_cg_mixed` for mixed strain/stress   macroscopic boundary conditions.